In [5]:
import argparse
import json
path = '/home/khh/workplace/LeakGFN/checkpoints/FM/seed_1/jnk3/20251229_180516'
json_file = f'{path}/args_info.json'
args_dict =json.load(open(json_file))
# args_dict['criterion'] = 'TB'
def dict_to_namespace(d):
    """Recursively convert a dictionary to argparse.Namespace."""
    ns = argparse.Namespace()
    for k, v in d.items():
        if isinstance(v, dict):
            setattr(ns, k, dict_to_namespace(v))
        else:
            setattr(ns, k, v)
    return ns

args = dict_to_namespace(args_dict)
args.objectives = args.objectives.split(',')
from gflownet.oracle.oracle import Oracle
from types import SimpleNamespace
oracle = Oracle(SimpleNamespace(
        objectives=args.objectives, 
        device=args.device
    ))

In [6]:
from gflownet.generator import LeakFMGFlowNet, TBGFlowNet, FMGFlowNet, MOReinforce, SubTBGFlowNet

# Now 'args' is similar to what you get from parser.parse_args()
if args.criterion == 'TB':
    generator = TBGFlowNet(args, args.bpath)
elif args.criterion == 'FM':
    generator = FMGFlowNet(args, args.bpath)
elif args.criterion == 'Reinforce':
    generator = MOReinforce(args, args.bpath)
elif args.criterion == 'SubTB':
    generator = SubTBGFlowNet(args, args.bpath)
else:
    raise ValueError(f'Unknown criterion: {args.criterion}')

# Set precision
if args.floatX == 'float64':
    generator = generator.double()

# Move to device
generator = generator.to(args.device)
print("FM : ", sum(p.numel() for p in generator.parameters() if p.requires_grad))

v4
FM :  1010794


In [7]:
import tqdm
import os
from train import ModelCheckpointer, RolloutWorker
import time
samples_list = []
iter = 10000
checkpoint_path = os.path.join(path, f'checkpoints/checkpoint_{iter}.pth')
checkpointer = ModelCheckpointer(path)
# _ = checkpointer.load_checkpoint(generator, checkpoint_path)
rollout_worker = RolloutWorker(args, args.bpath, oracle, 'cuda:0')
rollout_worker.min_blocks = 8

traj = []
start = time.time()
for i in tqdm.trange(1000):
    samples = rollout_worker.rollout(generator, False, replay=False)
    data = []
    if len(samples) != 8:
        continue
    samples_list.append(samples)
end = time.time()
print(end-start)

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [00:51<00:00, 19.58it/s]

51.076860427856445


In [8]:
from gflownet.generator import LeakFMGFlowNet, TBGFlowNet, FMGFlowNet, MOReinforce, SubTBGFlowNet
import argparse
import json
path = '/home/khh/workplace/LeakGFN/checkpoints/LeakGFN/seed_1/jnk3/20260113_235142'
json_file = f'{path}/args_info.json'
args_dict =json.load(open(json_file))
args_dict['criterion'] = 'LeakGFN'
args = dict_to_namespace(args_dict)
args.objectives = args.objectives.split(',')
# Now 'args' is similar to what you get from parser.parse_args()
if args.criterion == 'TB':
    generator = TBGFlowNet(args, args.bpath)
elif args.criterion == 'FM':
    generator = FMGFlowNet(args, args.bpath)
elif args.criterion == 'Reinforce':
    generator = MOReinforce(args, args.bpath)
elif args.criterion == 'SubTB':
    generator = SubTBGFlowNet(args, args.bpath)
elif args.criterion == 'LeakGFN':
    generator = LeakFMGFlowNet(args, args.bpath)
else:
    raise ValueError(f'Unknown criterion: {args.criterion}')

# Set precision
if args.floatX == 'float64':
    generator = generator.double()

# Move to device
generator = generator.to(args.device)
print("LeakGFN : ", sum(p.numel() for p in generator.parameters() if p.requires_grad))

v4
LeakGFN :  1234899


In [9]:
import tqdm
from train import ModelCheckpointer, RolloutWorker_Leak
samples_list = []
iter = 10000
checkpoint_path = os.path.join(path, f'checkpoints/checkpoint_{iter}.pth')
checkpointer = ModelCheckpointer(path)
# _ = checkpointer.load_checkpoint(generator, checkpoint_path)
rollout_worker = RolloutWorker_Leak(args, args.bpath, oracle, 'cuda:0')
rollout_worker.min_blocks = 8

traj = []
start = time.time()
for i in tqdm.trange(1000):
    samples = rollout_worker.rollout(generator, False, replay=False)
    data = []
    if len(samples) != 8:
        continue
    samples_list.append(samples)
end = time.time()
print(end-start)

100%|██████████| 1000/1000 [00:52<00:00, 19.04it/s]

52.520689964294434
